# Parte 1: Análise Exploratória e Predição de Demanda

Este notebook integra dados do 1746 com APIs de Clima e Feriados para entender os drivers de demanda municipal.

**Diferenciais Sênior:**
- Análise de Lag Features (t-1, t-2) para impacto climático.
- Modelo de Regressão Random Forest para volume diário (Treino 2023 / Teste 2024).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

# 1. Carregamento e Disclaimer
# NOTA: O dataset 'chamados_resumo' é sintético para fins de demonstração de pipeline.
df_chamados = pd.read_csv('../data/chamados_resumo_2023_2024.csv')
df_weather = pd.read_csv('../data/clima_rio_2023_2024.csv')
df_feriados = pd.read_csv('../data/feriados_rio_2023_2024.csv')

df_daily = df_chamados.groupby('data')['total_chamados'].sum().reset_index()
df_daily = df_daily.merge(df_weather, on='data', how='left')
df_daily['is_holiday'] = df_daily['data'].isin(df_feriados['data']).astype(int)

print(f"Dataset carregado: {df_daily.shape[0]} dias analisados.")

## 2. Feature Engineering: Lag Features (Q1 & Q3)
Criamos atrasos de 1 e 2 dias para precipitação para capturar o impacto tardio na demanda.

In [ ]:
df_daily['precip_lag1'] = df_daily['precipitation_sum'].shift(1)
df_daily['precip_lag2'] = df_daily['precipitation_sum'].shift(2)
df_daily.dropna(inplace=True) # Tratamento de NaN conforme auditoria

sns.heatmap(df_daily[['total_chamados', 'precipitation_sum', 'precip_lag1', 'precip_lag2']].corr(), annot=True)
plt.title('Correlação Demanda vs Clima (com Lags)')
plt.show()

## 4. Previsão de Demanda Multidimensional (Q4)
Modelo para prever o volume diário de chamados.

In [ ]:
df_daily['data'] = pd.to_datetime(df_daily['data'])
df_daily['weekday'] = df_daily['data'].dt.weekday

train = df_daily[df_daily['data'].dt.year == 2023]
test = df_daily[df_daily['data'].dt.year == 2024]

X_cols = ['temperature_2m_max', 'precipitation_sum', 'precip_lag1', 'is_holiday', 'weekday']
X_train, y_train = train[X_cols], train['total_chamados']
X_test, y_test = test[X_cols], test['total_chamados']

rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
print(f"MAE 2024: {mean_absolute_error(y_test, y_pred):.2f}")
print(f"R2 Score: {r2_score(y_test, y_pred):.2f}")

pd.Series(rf.feature_importances_, index=X_cols).sort_values().plot(kind='barh')
plt.title('Importância das Variáveis (Q4)')
plt.show()